# Deep Learning Model Evaluation & Error Analysis

**Concept**
Model analysis means looking *inside* the results of a trained neural network instead of only reading its accuracy. We compare training vs validation accuracy and loss to spot **overfitting** (train good, validation bad) and **underfitting** (both bad). Then we use the **confusion matrix**, **class-wise precision/recall/F1** and **misclassified samples** to see *which* classes fail and *why*.

**Why it is required / what problem it solves**
Accuracy alone is misleading: two models can have the same accuracy, but one may be overconfident and fragile. Error analysis tells us whether the fix is *more data*, *better features*, *regularization*, or a *different model size* - so we stop guessing.

In [1]:
# Beginner-friendly model analysis: train, then find out WHY it makes mistakes
import numpy as np
import warnings
warnings.filterwarnings('ignore')   # hide training warnings to keep output clean
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss, confusion_matrix, classification_report

# STEP 1: Load data (images of digits 0-9) and split into train / validation
X, y = load_digits(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5, random_state=42)

# Scale the pixel values so the network learns easily
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
print("STEP 1 - Train samples:", len(y_train), "| Validation samples:", len(y_val))

# STEP 2: Train 3 models of different sizes and compare train vs validation
settings = [("Too small", (2,), 0.0001),      # 2 neurons, very weak
            ("Too big", (256, 128), 0.0),     # huge, no regularization
            ("Balanced", (64,), 1.0)]         # medium, with regularization
print("\nSTEP 2 - Train vs Validation")
print("Model      TrainAcc ValAcc TrainLoss ValLoss  Diagnosis")
models = {}
for name, size, alpha in settings:
    model = MLPClassifier(hidden_layer_sizes=size, alpha=alpha, max_iter=60, random_state=1)
    model.fit(X_train, y_train)
    models[name] = model

    train_acc = model.score(X_train, y_train)
    val_acc = model.score(X_val, y_val)
    train_loss = log_loss(y_train, model.predict_proba(X_train))
    val_loss = log_loss(y_val, model.predict_proba(X_val))

    # Simple rules to diagnose the model
    if train_acc < 0.90 and val_acc < 0.90:
        diagnosis = "UNDERFITTING (both low)"
    elif val_loss > train_loss * 3:
        diagnosis = "OVERFITTING (val loss >> train loss)"
    else:
        diagnosis = "GOOD FIT"
    print("%-10s %8.3f %6.3f %9.3f %7.3f  %s" % (name, train_acc, val_acc, train_loss, val_loss, diagnosis))

# STEP 3: Look deeper at the Balanced model
best = models["Balanced"]
pred = best.predict(X_val)

print("\nSTEP 3 - Confusion matrix (rows = true digit, columns = predicted digit)")
print(confusion_matrix(y_val, pred))

print("\nSTEP 4 - Class-wise performance")
print(classification_report(y_val, pred))

# STEP 5: Misclassified samples
wrong = np.where(pred != y_val)[0]       # positions where prediction is wrong
print("STEP 5 - Wrong predictions:", len(wrong), "out of", len(y_val))
for i in wrong[:5]:
    confidence = best.predict_proba(X_val[i:i+1]).max()
    print("  Sample", i, "-> true:", y_val[i], "| predicted:", pred[i], "| confidence: %.2f" % confidence)

# STEP 6: Error patterns - which (true -> predicted) mistakes happen most?
cm = confusion_matrix(y_val, pred)
np.fill_diagonal(cm, 0)                  # ignore correct predictions
print("\nSTEP 6 - Most common mistakes:")
for _ in range(3):
    true_digit, pred_digit = np.unravel_index(cm.argmax(), cm.shape)
    print("  True", true_digit, "predicted as", pred_digit, ":", cm[true_digit, pred_digit], "times")
    cm[true_digit, pred_digit] = 0       # remove it so we find the next biggest

print("\nWhat the hyperparameters control:")
print("  hidden_layer_sizes -> model size (too small = underfit, too big = overfit)")
print("  alpha              -> penalty that keeps weights small (more alpha = less overfitting)")
print("  max_iter (epochs)  -> how many times the model sees the data")
print("  learning_rate_init -> size of each learning step (default 0.001)")


STEP 1 - Train samples: 898 | Validation samples: 899

STEP 2 - Train vs Validation
Model      TrainAcc ValAcc TrainLoss ValLoss  Diagnosis
Too small     0.332  0.333     1.691   1.705  UNDERFITTING (both low)
Too big       1.000  0.969     0.002   0.147  OVERFITTING (val loss >> train loss)
Balanced      0.992  0.954     0.097   0.186  GOOD FIT

STEP 3 - Confusion matrix (rows = true digit, columns = predicted digit)
[[82  0  0  0  0  0  0  0  0  0]
 [ 0 84  3  0  0  1  0  0  0  1]
 [ 0  0 82  1  0  0  0  0  0  0]
 [ 0  0  4 80  0  1  0  4  4  0]
 [ 0  0  0  0 92  0  0  1  0  0]
 [ 0  0  0  0  0 94  1  0  0  4]
 [ 0  0  0  0  2  0 96  0  0  0]
 [ 0  0  0  0  0  0  0 86  0  1]
 [ 0  3  0  0  0  1  0  1 76  2]
 [ 0  0  0  1  0  1  0  1  3 86]]

STEP 4 - Class-wise performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       0.97      0.94      0.95        89
           2       0.92      0.99      0.95        

**What this example tells us (Model Evaluation & Error Analysis)**
- **Too small** (2 neurons): train acc 0.33 and val acc 0.33 -> both low = **underfitting**.
- **Too big** (256x128, no penalty): train loss 0.002 but val loss 0.147 -> the model memorised the training data = **overfitting**, even though its val accuracy (0.969) looks great.
- **Balanced** (64 neurons, alpha=1.0): train loss 0.097 vs val loss 0.186 -> a small, healthy gap = **good fit**.
- Confusion matrix + class-wise F1: digit **3** is the weakest (recall 0.86). It is often confused with 2, 7 and 8 = **class confusion**. Wrong predictions still have 0.5-0.7 confidence, so the model is unsure on them.
- **Data-related causes:** only ~90 images per digit and tiny 8x8 pixels. **Model-related causes:** model too small or too big, and no penalty (alpha) against memorising.
- **Hyperparameters:** `hidden_layer_sizes` = model size, `alpha` = penalty that fights overfitting, `max_iter` = how many times the model sees the data, `learning_rate_init` = step size.

**AI/ML Example**
A digit classifier shows 95% accuracy, but the confusion matrix shows digit 3 is often mistaken for 2, 7 and 8. The fix is targeted: add more varied "3" images instead of simply training longer.

**Business Example**
A bank's fraud model is 99% accurate but misses most real fraud (low recall on the "fraud" class). Class-wise analysis reveals this, so the team adds more fraud examples and adjusts the decision threshold.

**AI/ML Engineer Use Case**
After every training run: compare train vs val loss, read the confusion matrix and per-class F1, and look at 20-50 wrong predictions. Then decide: add data (data problem), add a penalty or stop earlier (overfitting), or use a bigger model (underfitting). Save this as a standard report before deployment.